In [ ]:
import os
os.chdir('/home/asudupe/Latxa-Omni/')

In [2]:
import torch
from omni_speech.constants import SPEECH_TOKEN_INDEX, DEFAULT_SPEECH_TOKEN
from omni_speech.conversation import conv_templates, SeparatorStyle
from omni_speech.model.builder import load_pretrained_model
from omni_speech.datasets.preprocess import tokenizer_speech_token
from torch.utils.data import Dataset, DataLoader
import whisper
from datasets import load_dataset, load_from_disk
import numpy as np
from IPython.display import Audio
from scipy.io.wavfile import write
from torchaudio.transforms import Resample

/scratch/asudupe/conda-env/omni/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
dataset = load_from_disk('/scratch/asudupe/datasets/Instruct_S2S_eu')

In [32]:
speech = np.array(dataset['train'][10]['question_audio'], dtype=np.float32)
Audio(data=np.array(speech), rate=22050)

In [5]:
dataset['train'][0]['answer']

'Hona hemen osasuntsu egoteko hiru aholku: dieta orekatua jan, ur asko edanez hidratatuta egon eta gauero gutxienez zazpi ordu lo egin.'

In [6]:
write(filename='example.wav', data=np.array(dataset['train'][4]['question_audio'], dtype=np.float32), rate=22050)

In [13]:
speech_file = "omni_speech/serve/examples/helpful_base_1.wav"
speech = whisper.load_audio(speech_file)

Audio(data=np.array(speech), rate=16000)


In [3]:
# model_path = 'saves/13834/checkpoint-24000'
model_path = "/scratch/asudupe/checkpoints/Latxa-Llama-3.1-8B-Instruct/3493459/checkpoint-12000"
# model_path = "Llama-3.1-8B-Omni"
model_base = None
is_lora = False
s2s = False
mel_size = 128
conv_mode = 'llama_3'

In [4]:
tokenizer, model, context_len = load_pretrained_model(model_path, model_base, is_lora=is_lora, s2s=s2s)


/scratch/asudupe/conda-env/omni/lib/python3.10/site-packages/torch/_utils.py:831: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if you are using storages directly.  To access UntypedStorage directly, use tensor.untyped_storage() instead of tensor.storage()
  return self.fget.__get__(instance, owner)()
Loading checkpoint shards: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 4/4 [00:32<00:00,  8.01s/it]


In [14]:
speech_file = "audioak/voice_26-10-2025_01-02-00"
qs = "<speech>\nPlease directly answer the questions in the user's speech."
audio = whisper.load_audio(speech_file)
# audio = dataset['train'][20]['question_audio']
speech = torch.tensor(audio, dtype=torch.float32)
# speech = Resample(orig_freq=22050, new_freq=16000)(speech)

conv = conv_templates[conv_mode].copy()
conv.append_message(conv.roles[0], qs)
conv.append_message(conv.roles[1], None)
prompt = conv.get_prompt()

speech = whisper.pad_or_trim(speech)
speech = whisper.log_mel_spectrogram(speech, n_mels=mel_size).permute(1, 0)
input_ids = tokenizer_speech_token(prompt, tokenizer, return_tensors='pt')
speech_length = torch.LongTensor([speech.shape[0]])

input_ids = input_ids.to(device='cuda', non_blocking=True)
speech_tensor = speech.to(dtype=torch.float16, device='cuda', non_blocking=True)
speech_length = speech_length.to(device='cuda', non_blocking=True)

input_ids = torch.stack((input_ids, input_ids), dim=0)
speech_tensors = torch.stack((speech_tensor, speech_tensor), dim=0)
speech_lengths = torch.stack((speech_length, speech_length), dim=0)


In [15]:
Audio(data=np.array(audio), rate=16000)

In [16]:
temperature = 0.0
top_p = None
num_beams = 1
max_new_tokens = 256

with torch.inference_mode():
    outputs = model.generate(
        input_ids,
        speech=speech_tensors,
        speech_lengths=speech_lengths,
        do_sample=True if temperature > 0 else False,
        temperature=temperature,
        top_p=top_p,
        num_beams=num_beams,
        max_new_tokens=max_new_tokens,
        use_cache=True,
        pad_token_id=128004,
    )
output_ids = outputs

outputs = tokenizer.batch_decode(output_ids, skip_special_tokens=True)[0].strip()
outputs

/scratch/asudupe/conda-env/omni/lib/python3.10/site-packages/transformers/generation/configuration_utils.py:601: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.0` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(


'Munduko animaliarik handiena balea urdina da.'